<div dir="rtl">

# ۲ – ساخت RAG ساده روی اخبار فارسی

در این نوت‌بوک، زیرمجموعه‌ی داده‌ی آماده‌شده را به چند بخش کوچک (چانک) تبدیل می‌کنید، برای هر چانک بردارهای embedding می‌سازید، یک ایندکس برداری (مثلاً با FAISS) ایجاد می‌کنید و در نهایت یک سیستم RAG ساده برای پاسخ‌گویی به سؤال‌های فارسی پیاده‌سازی می‌کنید.

</div>


In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from typing import List, Tuple, Dict
import pickle
from pathlib import Path


In [ ]:
main_dir = "Desktop/BOOTCAMP/S9-RAG/پروژه  دوره RAG/rag-persian-news-lite-main"
df = pd.read_csv(f'{main_dir}/data/processed/news_subset.csv')
 
print(df.head()) 
print(df.info()) 
print(df['category'].value_counts())

In [ ]:

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> List[str]:
    
    if len(text) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text): 
        end = start + chunk_size 
        chunk = text[start:end]
         
        if end < len(text): 
            last_period = chunk.rfind('.') 
            last_space = chunk.rfind(' ') 

            break_point = max(last_period, last_space) 
            if break_point > chunk_size * 0.7:
                chunk = chunk[:break_point + 1]
                end = start + len(chunk)
         
        if chunk.strip():
            chunks.append(chunk.strip())
         
        if end < len(text):
            start = end - overlap
        else:
            break
    
    return chunks
 

chunks_data = []



In [ ]:

for idx, row in df.iterrows(): 
    text = row['text'] 
    doc_chunks = chunk_text(text, chunk_size=800, overlap=150) 
    for chunk_idx, chunk_text in enumerate(doc_chunks):
        chunks_data.append({
            'chunk_id': f"{row['doc_id']}_chunk_{chunk_idx}",
            'doc_id': row['doc_id'],
            'text': chunk_text,
            'category': row['category'],
            'date': row['date'],
            'title': row['title']
        })
      
chunks_df = pd.DataFrame(chunks_data)
 
print(chunks_df.head(10))


In [ ]:
# semantic embeddings
semantic_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
semantic_embeddings = semantic_model.encode(
    chunks_df['text'].tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True  # L2 normalization for better similarity
)
 
print(f"   Shape: {semantic_embeddings.shape}") 


# lexical embeddings
target_dim = semantic_embeddings.shape[1]

tfidf_vectorizer = TfidfVectorizer(
    max_features=target_dim,   
    ngram_range=(1, 2),   
    min_df=2,  #   document frequency
    max_df=0.8,   
    sublinear_tf=True   
)
 
lexical_embeddings_sparse = tfidf_vectorizer.fit_transform(chunks_df['text'].tolist())
 
lexical_embeddings = lexical_embeddings_sparse.toarray()
 
lexical_norms = np.linalg.norm(lexical_embeddings, axis=1, keepdims=True)
lexical_embeddings = lexical_embeddings / (lexical_norms + 1e-10)
 
print(f"   Shape: {lexical_embeddings.shape}") 

#  hybrid embeddings
alpha = 0.7  # 70% semantic, 30% lexical

print(f"   Weights: {alpha*100:.0f}% semantic + {(1-alpha)*100:.0f}% lexical")
 
hybrid_embeddings = alpha * semantic_embeddings + (1 - alpha) * lexical_embeddings 
hybrid_norms = np.linalg.norm(hybrid_embeddings, axis=1, keepdims=True)
hybrid_embeddings = hybrid_embeddings / (hybrid_norms + 1e-10) 
print(f"   Shape: {hybrid_embeddings.shape}")
 

In [ ]:

print(f"   Semantic: {semantic_embeddings.shape}")
print(f"   Lexical:  {lexical_embeddings.shape}")
print(f"   Hybrid:   {hybrid_embeddings.shape}")


In [ ]:
 
dimension = semantic_embeddings.shape[1]
 
semantic_index = faiss.IndexFlatIP(dimension)  
semantic_index.add(semantic_embeddings.astype('float32')) 
lexical_index = faiss.IndexFlatIP(dimension)
lexical_index.add(lexical_embeddings.astype('float32')) 
hybrid_index = faiss.IndexFlatIP(dimension)
hybrid_index.add(hybrid_embeddings.astype('float32')) 

In [ ]:
def retrieve(question: str, top_k: int = 5, embedding_model: str = 'hybrid') -> List[Dict]:
     
    if embedding_model == 'semantic':
        query_embedding = semantic_model.encode([question], normalize_embeddings=True)
        index = semantic_index
        
    elif embedding_model == 'lexical':
        query_vec = tfidf_vectorizer.transform([question]).toarray()
        query_norm = np.linalg.norm(query_vec)
        query_embedding = query_vec / (query_norm + 1e-10)
        index = lexical_index
        
    else:  # hybrid 
        sem_emb = semantic_model.encode([question], normalize_embeddings=True)
         
        lex_vec = tfidf_vectorizer.transform([question]).toarray()
        lex_norm = np.linalg.norm(lex_vec)
        lex_emb = lex_vec / (lex_norm + 1e-10) 
        query_embedding = alpha * sem_emb + (1 - alpha) * lex_emb
        query_norm = np.linalg.norm(query_embedding)
        query_embedding = query_embedding / (query_norm + 1e-10)
        
        index = hybrid_index
     
    scores, indices = index.search(query_embedding.astype('float32'), top_k)
     
    results = []
    for idx, score in zip(indices[0], scores[0]):
        chunk_info = chunks_df.iloc[idx]
        results.append({
            'chunk_idx': int(idx),
            'score': float(score),
            'chunk_id': chunk_info['chunk_id'],
            'doc_id': chunk_info['doc_id'],
            'title': chunk_info['title'],
            'category': chunk_info['category'],
            'date': chunk_info['date'],
            'text': chunk_info['text']
        })
    
    return results
 


In [ ]:
 
def answer(question: str, embedding_model: str = 'hybrid', top_k: int = 5) -> Dict: 
    retrieved_chunks = retrieve(question, top_k=top_k, embedding_model=embedding_model)
     
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[مستند {i} - امتیاز: {chunk['score']:.3f} - دسته: {chunk['category']}]\n"
            f"عنوان: {chunk['title']}\n"
            f"{chunk['text']}\n"
        )
    
    context = "\n".join(context_parts) 
    prompt = f"""بر اساس اسناد زیر به سوال پاسخ دهید:

{context}

سوال: {question}

پاسخ:""" 
    
    simple_answer = "\n\n".join([chunk['text'][:300] + "..." for chunk in retrieved_chunks[:3]])
    
    return {
        'question': question,
        'embedding_model': embedding_model,
        'top_k': top_k,
        'retrieved_chunks': retrieved_chunks,
        'context': context,
        'prompt': prompt,
        'answer': simple_answer
    }

In [ ]:
test_questions = [
    "آخرین اخبار فوتبال چیست؟",
    "در مورد اقتصاد ایران چه خبری هست؟",
    "اخبار سیاسی امروز چیست؟",
    "وضعیت آب و هوا چطور است؟"
]
 
for i, question in enumerate(test_questions, 1):
    print("\n" + "="*70)
    print(f"TEST {i}: {question}")
    print("="*70) 
    for model_type in ['semantic', 'lexical', 'hybrid']:
        print(f"\n--- Using {model_type.upper()} embeddings ---")
        
        result = answer(question, embedding_model=model_type, top_k=3)
        
        print(f"\n📄 Top 3 retrieved chunks:")
        for j, chunk in enumerate(result['retrieved_chunks'], 1):
            print(f"\n  {j}. [Score: {chunk['score']:.4f}] {chunk['title']}")
            print(f"     Category: {chunk['category']}")
            print(f"     Text preview: {chunk['text'][:150]}...")
        
        print(f"\n💡 Simple answer preview:")
        print(f"   {result['answer'][:200]}...")
 
print("\n\n" + "="*70)
print("COMPARISON: Average Scores for Each Embedding Type")
print("="*70)

comparison_question = "آخرین اخبار فوتبال چیست؟"

for model_type in ['semantic', 'lexical', 'hybrid']:
    result = answer(comparison_question, embedding_model=model_type, top_k=5)
    avg_score = np.mean([c['score'] for c in result['retrieved_chunks']])
    print(f"\n{model_type.upper()}:")
    print(f"  Average score: {avg_score:.4f}")
    print(f"  Top score: {result['retrieved_chunks'][0]['score']:.4f}")
    print(f"  Bottom score: {result['retrieved_chunks'][-1]['score']:.4f}")
 

Path(f'{main_dir}/data/processed').mkdir(parents=True, exist_ok=True) 
chunks_df.to_csv(f'{main_dir}/data/processed/chunks.csv', index=False, encoding='utf-8-sig')
 
faiss.write_index(semantic_index, f'{main_dir}data/processed/semantic_index.faiss')
faiss.write_index(lexical_index, f'{main_dir}data/processed/lexical_index.faiss')
faiss.write_index(hybrid_index, f'{main_dir}data/processed/hybrid_index.faiss')
 
with open('../data/processed/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
 
with open('../data/processed/alpha.txt', 'w') as f:
    f.write(str(alpha))
print(f"""
 Chunking complete: {len(chunks_df)} chunks from {len(df)} documents
 Embeddings created:
   - Semantic: {semantic_embeddings.shape}
   - Lexical: {lexical_embeddings.shape}
   - Hybrid: {hybrid_embeddings.shape}
 FAISS indices built:
   - Semantic: {semantic_index.ntotal} vectors
   - Lexical: {lexical_index.ntotal} vectors
   - Hybrid: {hybrid_index.ntotal} vectors
 Functions ready:
   - retrieve(question, top_k, embedding_model)
   - answer(question, embedding_model, top_k)
 All artifacts saved to '../data/processed/'

""")
